In [13]:
!pip install pyvi pycocoevalcap pandas

In [14]:
import json
import os
import pandas as pd
from pyvi import ViTokenizer
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge
from collections import defaultdict

# --- 1. CẤU HÌNH ĐƯỜNG DẪN FILE ---
GT_FILE = "val_groundtruth.jsonl"
PRED_FILES = {
    "Qwen2-VL-FT-E1 (step 2000)": "results_qwen2vl_val_E1.jsonl",
    "Qwen2-VL-FT-E2 (step 4000)": "results_qwen2vl_val_E2.jsonl",
    "Qwen2-VL-FT-E3 (step 6000)": "results_qwen2vl_val_E3.jsonl"
}

In [15]:
# --- 2. HÀM TIỀN XỬ LÝ VÀ LOAD DATA ---
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def preprocess_vn(text):
    if not text: return ""
    return ViTokenizer.tokenize(text.lower())

# A. Load Groundtruth và gom nhóm (vì GT lộn xộn và 1 ảnh có 5 câu)
print("📂 Đang nạp Groundtruth...")
gt_raw = load_jsonl(GT_FILE)
references = defaultdict(list)
for item in gt_raw:
    img_id = item['file_name']
    references[img_id].append(preprocess_vn(item['caption']))

# B. Hàm tính toán Metrics cho một file dự đoán
def eval_model(pred_path):
    preds_raw = load_jsonl(pred_path)
    
    gts = {}
    res = {}
    
    for item in preds_raw:
        img_id = item['file_name']
        if img_id in references:
            # Tokenize dự đoán
            res[img_id] = [preprocess_vn(item['prediction'])]
            # Lấy nhãn tương ứng đã tokenize sẵn
            gts[img_id] = references[img_id]
            
    # Khởi tạo bộ chấm điểm
    scorers = [
        (Cider(), "CIDEr"),
        (Rouge(), "ROUGE-L")
    ]
    
    results = {}
    for scorer, method in scorers:
        score, _ = scorer.compute_score(gts, res)
        if isinstance(method, list):
            for m, s in zip(method, score):
                results[m] = round(s * 100, 2)
        else:
            results[method] = round(score * 100, 2)
    return results

📂 Đang nạp Groundtruth...


In [ ]:
# --- 3. THỰC THI ĐÁNH GIÁ TẤT CẢ MODEL ---
all_metrics = {}

for model_name, path in PRED_FILES.items():
    print(f"📊 Đang chấm điểm cho: {model_name}...")
    all_metrics[model_name] = eval_model(path)


# Kết hợp Metrics và Efficiency
final_comparison = {}
for name in PRED_FILES.keys():
    final_comparison[name] = {**all_metrics[name]}

# --- 5. XUẤT BẢNG TỔNG HỢP ---
df = pd.DataFrame(final_comparison).T

# Sắp xếp cột cho đúng thứ tự mong muốn
cols = ["CIDEr", "ROUGE-L"]
df = df[cols]

print("\n" + "="*100)
print("🏆 BẢNG SO SÁNH HIỆU NĂNG MODEL SAU TƯNG EPOCH (WORD-LEVEL)")
print("="*100)
display(df)
print("="*100)

📊 Đang chấm điểm cho: Qwen2-VL-FT-E1 (step 2000)...
📊 Đang chấm điểm cho: Qwen2-VL-FT-E2 (step 4000)...
📊 Đang chấm điểm cho: Qwen2-VL-FT-E3 (step 6000)...

🏆 BẢNG SO SÁNH HIỆU NĂNG MODEL SAU TƯNG EPOCH (WORD-LEVEL)


,CIDEr,ROUGE-L
Qwen2-VL-FT-E1 (step 2000),43.34,39.00
Qwen2-VL-FT-E2 (step 4000),46.60,39.09
Qwen2-VL-FT-E3 (step 6000),46.09,38.83
